# Full Reserving Analysis: End-to-End Template

**Starting point:** Two `chainladder.Triangle` objects — paid and reported/incurred cumulative losses.  
**Worked example:** `cl.load_sample('quarterly')` — 12 accident years (1995–2006), quarterly development (3, 6, 9 … 135 months), valuation date 2006-Q1.

> **To use with your own data:** replace the `paid_triangle` and `reported_triangle` assignments in Section 0 with your own Triangle objects. Every cell below adapts automatically.

## Table of Contents
0. Setup & Input Data  
1. EDA — Triangle Quality & Maturity Profile  
2. Pattern Diagnostics — Is the Triangle Well-Behaved?  
3. Data Adjustments (Berquist-Sherman) — When and How  
4. Development Pattern Selection + Cross-Validation  
5. Tail Selection  
6. Apply All Methods  
7. Credibility Blending (VotingChainladder)  
8. Actual vs Expected — One-Period Emergence Test  
9. Reserve Range & Sensitivity  
10. Selected Reserve & Documentation  


## Section 0 — Setup & Input Data

Replace the two triangle assignments below with your own data.  
All sections downstream consume `paid_triangle` and `reported_triangle` only.

**Requirements:**
- Both triangles must be **cumulative** (`triangle.is_cumulative == True`).  
- Both must cover the same accident-year origins and development ages.  
- Development ages in **months**: 3, 6, 9, … for quarterly grain; 12, 24, 36, … for annual.


In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import chainladder as cl

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import (
    build_exposure_triangle,
    calendar_year_diagnostic,
    link_ratio_table,
    paid_vs_incurred_comparison,
    plot_link_ratio_heatmap,
    trend_summary,
    run_chain_ladder,
    run_bornhuetter_ferguson,
    run_benktander,
    run_cape_cod,
    run_case_outstanding_chainladder,
    run_expected_loss,
)

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
pd.set_option('display.max_columns', 20)

# ─────────────────────────────────────────────────────────────────────────────
# REPLACE THESE TWO LINES WITH YOUR OWN TRIANGLE OBJECTS
# ─────────────────────────────────────────────────────────────────────────────
_raw = cl.load_sample('quarterly')
paid_triangle     = _raw['paid']       # cumulative paid losses
reported_triangle = _raw['incurred']   # cumulative reported (paid + case reserves)
# ─────────────────────────────────────────────────────────────────────────────

print('Triangle grain:   quarterly (3-month development steps)')
print('Paid shape:       ', paid_triangle.shape)
print('Reported shape:   ', reported_triangle.shape)
print('Accident years:   ', list(paid_triangle.origin.year))
print('Development ages: ', list(paid_triangle.development[:8]), '...')
print('Valuation date:   ', paid_triangle.valuation_date.strftime('%Y-%m-%d'))


## Section 1 — EDA: Triangle Quality & Maturity Profile

**Goal:** understand the data before fitting anything.  
Check for: completeness gaps, maturity distribution, paid-to-reported relationship at latest diagonal.

**% Paid = paid / reported** at the latest diagonal is the most important maturity indicator.  
A very low % paid (< 20%) means the development pattern has high leverage and the chain ladder will be unreliable for that accident year — BF or Benktander will be more appropriate.


In [ ]:
# ── 1a. Maturity profile ────────────────────────────────────────────────────
latest_paid_df = (
    paid_triangle.latest_diagonal
    .to_frame(origin_as_datetime=False, keepdims=True)
)
latest_rep_df = (
    reported_triangle.latest_diagonal
    .to_frame(origin_as_datetime=False, keepdims=True)
)

latest_age = (
    paid_triangle.to_frame(origin_as_datetime=False)
    .apply(lambda r: r.dropna().index[-1] if r.notna().any() else None, axis=1)
)

# Rough CDF estimate (no tail) to get % reported / % unreported
_dev_cdf = cl.Development(average='volume', n_periods=-1).fit(reported_triangle).cdf_.to_frame().iloc[0]

def _cdf_for_age(age):
    key = f'{int(age)}-Ult'
    return float(_dev_cdf.get(key, 1.0)) if pd.notna(age) else np.nan

maturity = pd.DataFrame({
    'latest_age':   latest_age.values,
    'paid':         latest_paid_df['paid'].values,
    'reported':     latest_rep_df['incurred'].values,
}, index=latest_paid_df['origin'].values)

maturity['pct_paid']      = maturity['paid'] / maturity['reported']
maturity['case_reserve']  = maturity['reported'] - maturity['paid']
maturity['approx_cdf']    = maturity['latest_age'].map(_cdf_for_age)
maturity['pct_unreported']= 1.0 - 1.0 / maturity['approx_cdf'].replace(0, np.nan)

print('Maturity profile at latest diagonal:')
maturity.style.format({
    'paid':           '{:,.0f}',
    'reported':       '{:,.0f}',
    'case_reserve':   '{:,.0f}',
    'pct_paid':       '{:.1%}',
    'approx_cdf':     '{:.3f}',
    'pct_unreported': '{:.1%}',
})


In [ ]:
# ── 1b. Incremental heatmaps ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_link_ratio_heatmap(paid_triangle,     ax=axes[0], title='Paid — link ratio heatmap')
plot_link_ratio_heatmap(reported_triangle, ax=axes[1], title='Reported — link ratio heatmap')
plt.suptitle('Diagnostic: green = fast development, red = slow development', y=1.01)
plt.tight_layout()
plt.show()


## Section 2 — Pattern Diagnostics: Is the Triangle Well-Behaved?

Four screens inform whether the raw triangle is suitable for direct pattern fitting,
or whether adjustments are needed before selection.

| Screen | Tool | What to look for |
|---|---|---|
| 1. Link ratio exhibit | `link_ratio_table` | Divergence between averaging methods; outlier AYs |
| 2. Calendar-year (L/S) test | `calendar_year_diagnostic` | Systematic above/below-average development by calendar year |
| 3. Paid vs reported LDF | `paid_vs_incurred_comparison` | Trend in ldf_ratio → settlement rate or case adequacy shift |
| 4. Development trend | `trend_summary` | Trending factors across AYs → recency weighting needed |


In [ ]:
# ── Screen 1: Link ratio exhibit ────────────────────────────────────────────
lrt = link_ratio_table(paid_triangle, n_periods=-1)
print('Link ratio exhibit — paid triangle (first 14 development transitions):')
print(lrt.iloc[:, :14].to_string())


In [ ]:
# ── Screen 2: Calendar-year L/S diagnostic ──────────────────────────────────
cal_diag   = calendar_year_diagnostic(paid_triangle)
cy_summary = cal_diag['calendar_year_summary']
print('Calendar-year L/S summary:')
print(cy_summary.to_string())
print()
print('  pct_L > 0.65 -> above-average development (inflation, reserve releases)')
print('  pct_L < 0.35 -> below-average development (reserve strengthening, slow payment)')
print('  Flagged calendar years are candidates for drop_valuation exclusion.')


In [ ]:
# ── Screen 3: Paid vs reported LDF comparison ────────────────────────────────
pvi = paid_vs_incurred_comparison(paid_triangle, reported_triangle)
print('LDF comparison (paid vs reported):')
print(pvi['ldf_comparison'].to_string())
print()
print('Paid-to-reported ratio at latest diagonal:')
print(pvi['paid_to_incurred_latest'].to_string())
print()
print('ldf_ratio trending UP   -> settlement rate acceleration (paid speeding up)')
print('ldf_ratio trending DOWN -> case reserve strengthening (reported slowing)')
print('Either trend may warrant a Berquist-Sherman adjustment (Section 3).')


In [ ]:
# ── Screen 4: Development trend summary ─────────────────────────────────────
try:
    ts = trend_summary(paid_triangle)
    print('Slope of link ratios across accident years (negative = factors declining over time):')
    print(ts['age_trends'].to_frame('slope_across_AYs').T.iloc[:, :14].to_string())
    print()
    print('Diagonal trend (slope of average link ratio by calendar year):')
    print(ts['diagonal_trend'].to_string())
except Exception as e:
    print(f'trend_summary not available for this triangle: {e}')
    print('(trend_summary requires a minimum of two complete accident years with full development history)')


### Decision Gate

| Finding | Recommended action |
|---|---|
| No anomalies | Proceed directly to Section 4 |
| Anomalous calendar-year diagonal(s) | Use `drop_valuation` parameter in Section 4 |
| Paid/reported LDF ratio trending | Run Berquist-Sherman in Section 3 |
| Factors clearly trending across AYs | Use `n_periods=3` or `n_periods=5` in Section 4 |

For this worked example, Section 3 shows the Berquist-Sherman template; no adjustment is applied since the `quarterly` dataset lacks count triangles.


## Section 3 — Data Adjustments: Berquist-Sherman (if needed)

**When to use:** Screen 3 above shows a material trend in the paid/reported LDF ratio.  
- **Upward trend** → settlement-rate acceleration → paid pattern understates future payments  
- **Downward trend** → case reserve strengthening → reported pattern overstates future development

**B-S requires count triangles** (`closed_count`, `reported_count`) in addition to paid and incurred.  
The `quarterly` dataset does not include count data, so the full adjustment cannot be demonstrated.  
The template below shows the exact call structure for when count data is available.


In [ ]:
# ── Berquist-Sherman template ────────────────────────────────────────────────
# Uncomment and populate with your count triangles when Screen 3 flags a shift.
#
# bs = cl.BerquistSherman(
#     paid_amount     = paid_triangle,
#     incurred_amount = reported_triangle,
#     reported_count  = reported_count_triangle,   # cumulative reported (open) counts
#     closed_count    = closed_count_triangle,     # cumulative closed counts
#     trend           = 0.05,   # assumed annual settlement trend; positive = accelerating
# )
# bs.fit(paid_triangle)
# paid_adjusted = bs.adjusted_triangle_
#
# Compare LDFs before and after:
# lrt_orig = link_ratio_table(paid_triangle)
# lrt_adj  = link_ratio_table(paid_adjusted)
# pd.DataFrame({
#     'original':   lrt_orig.iloc[-2],
#     'adjusted':   lrt_adj.iloc[-2],
#     'difference': lrt_adj.iloc[-2] - lrt_orig.iloc[-2],
# })
#
# If adjustment is applied, replace paid_triangle going forward:
# paid_triangle = paid_adjusted

print('Berquist-Sherman: no adjustment applied for this dataset.')
print('See comments above for the template when closed_count and reported_count are available.')


## Section 4 — Development Pattern Selection

Pattern selection combines three inputs:

1. **Empirical candidates** — compare all standard averaging methods side-by-side  
2. **n_periods sensitivity** — identify whether recent AYs differ from older ones  
3. **Cross-validation scoring** — rank configurations by holdout emergence accuracy

The cross-validation step is the most objective input: it selects the development assumption that was most predictive of actual recent emergence, not just the one that produces the largest or smallest reserve.


In [ ]:
# ── 4a. All averaging methods compared ──────────────────────────────────────
specs = {
    'volume (all)':      cl.Development(average='volume',  n_periods=-1),
    'simple (all)':      cl.Development(average='simple',  n_periods=-1),
    'volume (last 5)':   cl.Development(average='volume',  n_periods=5),
    'volume (last 3)':   cl.Development(average='volume',  n_periods=3),
    'drop hi+lo':        cl.Development(average='simple',  n_periods=-1, drop_high=True, drop_low=True),
}

ldf_rows = {}
for name, m in specs.items():
    try:
        ldf_rows[name] = m.fit(paid_triangle).ldf_.to_frame().iloc[0]
    except Exception:
        pass

# Geometric (manual)
lr_frame = paid_triangle.link_ratio.to_frame()
geo = np.exp(np.log(lr_frame.clip(lower=1e-9)).mean())
geo.name = 'geometric (all)'

ldf_table = pd.DataFrame(ldf_rows).T
ldf_table = pd.concat([ldf_table, geo.to_frame().T])
ldf_table.index.name = 'method'

print('LDF comparison — first 14 transitions:')
ldf_table.iloc[:, :14].style.format('{:.4f}').background_gradient(axis=0, cmap='Blues')


In [ ]:
# ── 4b. n_periods sensitivity ────────────────────────────────────────────────
n_options  = [-1, 7, 5, 3]
labels     = ['All', 'Last 7', 'Last 5', 'Last 3']
key_trans  = ['3-6', '6-9', '9-12', '12-15']

fig, axes = plt.subplots(1, len(key_trans), figsize=(14, 4))
for ax, trans in zip(axes, key_trans):
    vals = []
    for n in n_options:
        try:
            ldf = cl.Development(average='volume', n_periods=n).fit(paid_triangle).ldf_.to_frame().iloc[0]
            vals.append(ldf.get(trans, np.nan))
        except Exception:
            vals.append(np.nan)
    ax.plot(labels, vals, marker='o', color='steelblue')
    ax.set_title(f'LDF {trans}')
    ax.set_ylabel('LDF' if trans == key_trans[0] else '')
    ax.grid(True, alpha=0.3)
    for lbl, v in zip(labels, vals):
        if not np.isnan(v):
            ax.annotate(f'{v:.3f}', (lbl, v), textcoords='offset points', xytext=(0, 7),
                        ha='center', fontsize=8)

plt.suptitle('Volume-weighted LDF vs n_periods — key early transitions', y=1.03)
plt.tight_layout()
plt.show()
print('A steep slope means factors are trending; use lower n_periods for those transitions.')


In [ ]:
# ── 4c. Industry benchmark overlay ──────────────────────────────────────────
vol_ldfs = cl.Development(average='volume', n_periods=-1).fit(paid_triangle).ldf_.to_frame().iloc[0]
# Simulate an external benchmark: volume-weighted + 2% conservatism adjustment
benchmark_ldfs = {int(col.split('-')[0]): round(float(v) * 1.02, 4) for col, v in vol_ldfs.items()}
dev_bench = cl.DevelopmentConstant(patterns=benchmark_ldfs, style='ldf').fit(paid_triangle)
ldf_bench = dev_bench.ldf_.to_frame().iloc[0]

print('Benchmark vs empirical volume-weighted (first 10 transitions):')
pd.DataFrame({
    'Benchmark (+2% adjustment)': ldf_bench,
    'Volume-weighted (all AYs)':  vol_ldfs,
    'Difference':                  ldf_bench - vol_ldfs,
}).iloc[:10]


### Cross-Validation: Scoring Development Configurations by Holdout Emergence Accuracy

**Method:**
1. Truncate the triangle to remove the most recent complete diagonal (one quarter of data).  
2. Fit a Pipeline on the truncated triangle for each configuration in the parameter grid.  
3. Use `model.full_expectation_` to predict the expected cumulative at each AY's held-out age.  
4. Compare to the actual value from the full triangle.  
5. Rank configurations by Mean Absolute Error (lower = better).

**Why this is better than scoring by IBNR alone:** configurations that produce extreme reserves are often those that fit the past poorly; the holdout test provides direct empirical evidence about predictive accuracy.


In [ ]:
# ── 4d. Cross-validation ─────────────────────────────────────────────────────
latest_val = paid_triangle.valuation_date
cutoff_val = latest_val - pd.DateOffset(months=3)   # hold out one quarter
paid_trunc = paid_triangle[paid_triangle.valuation <= cutoff_val]

print(f'Full triangle valuation:  {latest_val.strftime("%Y-%m-%d")}')
print(f'Truncated to:             {cutoff_val.strftime("%Y-%m-%d")}')

# Pre-compute held-out (origin, held_out_age, actual_value) for scoring
full_frame  = paid_triangle.to_frame(origin_as_datetime=False)
trunc_frame = paid_trunc.to_frame(origin_as_datetime=False)

holdout_pairs = {}  # {origin: (held_out_age, actual_value)}
for origin in trunc_frame.index:
    row = trunc_frame.loc[origin].dropna()
    if len(row) == 0:
        continue
    held_out_age = row.index[-1] + 3
    if held_out_age in full_frame.columns:
        actual = full_frame.loc[origin, held_out_age]
        if pd.notna(actual):
            holdout_pairs[origin] = (held_out_age, float(actual))

print(f'AYs with a held-out observation: {len(holdout_pairs)}')

# Scorer: MAE on held-out diagonal (negate because GridSearch maximizes)
def score_holdout_mae(fitted_pipe):
    model  = fitted_pipe.named_steps['model']
    fe_df  = model.full_expectation_.to_frame(origin_as_datetime=False)
    errors = []
    for origin, (age, actual) in holdout_pairs.items():
        if origin in fe_df.index and age in fe_df.columns:
            exp = fe_df.loc[origin, age]
            if pd.notna(exp):
                errors.append(abs(float(exp) - actual))
    return -float(np.mean(errors)) if errors else -np.inf

pipe_cv = cl.Pipeline([
    ('dev',   cl.Development()),
    ('tail',  cl.TailCurve()),
    ('model', cl.Chainladder()),
])
param_grid_cv = {
    'dev__average':   ['volume', 'simple'],
    'dev__n_periods': [3, 5, 7, -1],
    'tail__curve':    ['exponential', 'inverse_power'],
}

grid_cv = cl.GridSearch(pipe_cv, param_grid=param_grid_cv, scoring=score_holdout_mae)
grid_cv.fit(paid_trunc)

cv_results = (
    grid_cv.results_
    .rename(columns={'score': 'neg_mae'})
    .assign(holdout_mae=lambda df: -df['neg_mae'])
    .sort_values('holdout_mae')
    .reset_index(drop=True)
)
cv_results.index += 1
cv_results.index.name = 'rank'

print('Cross-validation results (ranked by holdout MAE — lower = better):')
cv_results.style.format({'holdout_mae': '{:.2f}'}).background_gradient(
    subset=['holdout_mae'], cmap='RdYlGn_r'
)


In [ ]:
# ── 4e. Pattern selection rationale ─────────────────────────────────────────
best_row  = cv_results.iloc[0]
sel_avg   = best_row['dev__average']
sel_n     = int(best_row['dev__n_periods'])
sel_tail  = best_row['tail__curve']

print(f'Selected configuration (cross-validation winner):')
print(f'  average   = {sel_avg!r}')
print(f'  n_periods = {sel_n}')
print(f'  tail      = {sel_tail!r}')
print(f'  holdout MAE = {best_row["holdout_mae"]:.2f}')
print()

# Fit selected development on the FULL triangle
selected_dev = cl.Development(average=sel_avg, n_periods=sel_n).fit_transform(paid_triangle)
selected_ldfs = cl.Development(average=sel_avg, n_periods=sel_n).fit(paid_triangle).ldf_.to_frame().iloc[0]

print('Selected LDFs (first 16 transitions):')
selected_ldfs.iloc[:16].to_frame('selected_ldf').T


## Section 5 — Tail Selection

The tail factor covers development beyond the last observed age in the triangle.  
For quarterly triangles extending to 135 months (11.25 years), the tail can be small for short/medium-tail lines but still materially affects immature AYs.

| Estimator | Mechanism | Best for |
|---|---|---|
| `TailCurve(exponential)` | Fits $f_d = e^{-\theta d}$; smooth extrapolation | Short/medium-tail lines |
| `TailCurve(inverse_power)` | Fits $f_d = d^{-\theta}$; heavier tail | Long-tail lines (GL, WC) |
| `TailBondy` | $\text{tail} = f_{\text{last}}^{b}$; self-referencing | Any line; no external input needed |
| `TailConstant` | Fixed factor beyond last age | When an industry benchmark is available |


In [ ]:
# ── 5a. Fit all tail options ─────────────────────────────────────────────────
tail_candidates = {
    'TailCurve(exp)':     cl.TailCurve(curve='exponential'),
    'TailCurve(pow)':     cl.TailCurve(curve='inverse_power'),
    'TailBondy':          cl.TailBondy(),
    'TailConstant 1.000': cl.TailConstant(tail=1.000),
    'TailConstant 1.010': cl.TailConstant(tail=1.010),
    'TailConstant 1.030': cl.TailConstant(tail=1.030),
}

tail_rows = []
for name, tail_est in tail_candidates.items():
    try:
        t          = tail_est.fit(selected_dev)
        tail_f     = float(t.tail_.iloc[0, 0])
        tailed     = tail_est.fit_transform(selected_dev)
        total_ibnr = float(np.nansum(cl.Chainladder().fit(tailed).ibnr_.values))
        tail_rows.append({'Tail': name, 'TailFactor': tail_f, 'TotalIBNR': total_ibnr})
    except Exception as e:
        tail_rows.append({'Tail': name, 'TailFactor': np.nan, 'TotalIBNR': np.nan})

tail_df = pd.DataFrame(tail_rows).set_index('Tail')
print('Tail factor comparison:')
tail_df.style.format({'TailFactor': '{:.5f}', 'TotalIBNR': '{:,.0f}'})


In [ ]:
# ── 5b. IBNR sensitivity to tail choice ─────────────────────────────────────
valid = tail_df.dropna()
if len(valid) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    bars   = ax.barh(valid.index, valid['TotalIBNR'], color=colors[:len(valid)], alpha=0.85)
    ax.set_xlabel('Total IBNR')
    ax.set_title('IBNR Sensitivity to Tail Choice')
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    for bar, val in zip(bars, valid['TotalIBNR']):
        ax.text(bar.get_width() * 1.005, bar.get_y() + bar.get_height() / 2,
                f'{val:,.0f}', va='center', fontsize=9)
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Select tail from cross-validation result
selected_tail     = cl.TailCurve(curve=sel_tail)
tailed_dev        = selected_tail.fit_transform(selected_dev)
selected_tail_f   = float(selected_tail.fit(selected_dev).tail_.iloc[0, 0])
print(f'Selected tail: TailCurve(curve={sel_tail!r})  ->  tail factor = {selected_tail_f:.5f}')


## Section 6 — Apply All Methods

All methods receive the same development pattern and tail from Sections 4–5.  
This ensures the comparison reflects only method differences, not pattern differences.

**Exposure proxy:** earned premium is back-solved from latest paid / assumed loss ratio.  
Replace with actual earned premium when available — this directly affects BF, Benktander, Cape Cod.

| Method | When to prefer |
|---|---|
| Chain Ladder | Stable development, mature AYs, no credibility concerns |
| Expected Loss Ratio | No credible triangle data; pure a priori estimate |
| Bornhuetter-Ferguson | Immature AYs; credible a priori available |
| Benktander (n=2) | Blend BF and CL; attenuates a priori dependence |
| Cape Cod | Self-calibrating ECR; premium data available and on-level |
| Case Outstanding | Separate paid from case reserve development |


In [ ]:
# ── 6a. Build exposure ────────────────────────────────────────────────────────
apriori_lr = 0.75

latest_paid_vals = (
    paid_triangle.latest_diagonal
    .to_frame(origin_as_datetime=False, keepdims=True)['paid'].values
)
ay_years = np.array([
    int(str(x)[:4])
    for x in paid_triangle.latest_diagonal
    .to_frame(origin_as_datetime=False, keepdims=True)['origin'].values
])

ep_proxy       = latest_paid_vals / apriori_lr
exposure_series = pd.Series(ep_proxy.astype(float), index=ay_years, dtype=float)

print('Exposure (earned premium proxy) by accident year:')
pd.DataFrame({
    'LatestPaid':      latest_paid_vals,
    'EarnedPremProxy': ep_proxy,
    'ImpliedLR':       latest_paid_vals / ep_proxy,
}, index=ay_years)


In [ ]:
# ── 6b. Run all methods ───────────────────────────────────────────────────────
dev_kw  = dict(average=sel_avg, n_periods=sel_n)
tail_kw = dict(curve=sel_tail)

methods = {}

res_cl, summ_cl, _ = run_chain_ladder(
    paid_triangle, development_kwargs=dev_kw, tail_kind='curve', tail_kwargs=tail_kw)
methods['Chain Ladder'] = (res_cl, summ_cl)

res_el, summ_el, _ = run_expected_loss(
    paid_triangle, exposure_series, apriori=apriori_lr,
    development_kwargs=dev_kw, tail_kind='curve', tail_kwargs=tail_kw)
methods['Expected Loss Ratio'] = (res_el, summ_el)

res_bf, summ_bf, _ = run_bornhuetter_ferguson(
    paid_triangle, exposure_series, apriori=apriori_lr,
    development_kwargs=dev_kw, tail_kind='curve', tail_kwargs=tail_kw)
methods['BF'] = (res_bf, summ_bf)

res_bk, summ_bk, _ = run_benktander(
    paid_triangle, exposure_series, apriori=apriori_lr, n_iters=2,
    development_kwargs=dev_kw, tail_kind='curve', tail_kwargs=tail_kw)
methods['Benktander (n=2)'] = (res_bk, summ_bk)

res_cc, summ_cc, _ = run_cape_cod(
    paid_triangle, exposure_series,
    development_kwargs=dev_kw, tail_kind='curve', tail_kwargs=tail_kw)
methods['Cape Cod'] = (res_cc, summ_cc)

res_co, summ_co, _ = run_case_outstanding_chainladder(
    paid_triangle, reported_triangle,
    paid_n_periods=sel_n, case_n_periods=sel_n)
methods['Case Outstanding'] = (res_co, summ_co)

print('All methods run successfully:')
for name, (res, _) in methods.items():
    print(f'  {name:<25} IBNR = {res.ibnr_total:>10,.0f}   Ultimate = {res.ultimate_total:>10,.0f}')


In [ ]:
# ── 6c. AY-level ultimate comparison ─────────────────────────────────────────
ay_ult = {name: summ['ultimate'] for name, (_, summ) in methods.items() if 'ultimate' in summ.columns}
ult_table = pd.DataFrame(ay_ult)
ult_table['Low']   = ult_table.min(axis=1)
ult_table['High']  = ult_table.max(axis=1)
ult_table['Range'] = ult_table['High'] - ult_table['Low']
print('Ultimate by accident year — larger Range = more judgment-sensitive AY:')
ult_table.style.format('{:,.0f}').background_gradient(subset=['Range'], cmap='YlOrRd')


In [ ]:
# ── 6d. IBNR comparison chart ─────────────────────────────────────────────────
ibnr_data = {name: summ['ibnr'] for name, (_, summ) in methods.items() if 'ibnr' in summ.columns}
ibnr_df   = pd.DataFrame(ibnr_data)

ax = ibnr_df.plot(kind='bar', figsize=(13, 5), colormap='tab10', alpha=0.85,
                  title='IBNR by Accident Year — All Methods')
ax.set_xlabel('Origin Year')
ax.set_ylabel('IBNR')
ax.legend(loc='upper left', fontsize=8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Total IBNR by method (sorted high to low):')
ibnr_df.sum().sort_values(ascending=False).map('{:,.0f}'.format)


## Section 7 — Credibility Blending (VotingChainladder)

`VotingChainladder` combines predictions from multiple estimators using an explicit weight matrix.

**Weight logic below:**
- AYs with % paid > 70% → mature → weight Chain Ladder heavily (0.7 CL / 0.2 BF / 0.1 CC)  
- AYs with % paid 40–70% → developing → balanced blend (0.4 / 0.3 / 0.3)  
- AYs with % paid < 40% → immature → rely on BF/Cape Cod (0.1 / 0.5 / 0.4)  

Adjust the thresholds and weights to match your view of each AY's credibility.


In [ ]:
# ── 7a. Weight matrix by maturity ────────────────────────────────────────────
pct_paid   = maturity['pct_paid'].fillna(0.0).values
n_origins  = len(pct_paid)

# Estimator order: [CL, BF, Cape Cod]
W = np.array([
    [0.7, 0.2, 0.1] if p > 0.70 else
    [0.4, 0.3, 0.3] if p > 0.40 else
    [0.1, 0.5, 0.4]
    for p in pct_paid
])

weight_df = pd.DataFrame(W, columns=['CL', 'BF', 'Cape Cod'],
                          index=maturity.index[:n_origins])
weight_df.insert(0, 'pct_paid', pct_paid)
print('Blending weight matrix:')
weight_df.style.format({'pct_paid': '{:.1%}', 'CL': '{:.1f}', 'BF': '{:.1f}', 'Cape Cod': '{:.1f}'})


In [ ]:
# ── 7b. VotingChainladder ─────────────────────────────────────────────────────
pipe_cl = cl.Pipeline([
    ('dev',   cl.Development(average=sel_avg, n_periods=sel_n)),
    ('tail',  cl.TailCurve(curve=sel_tail)),
    ('model', cl.Chainladder()),
])
pipe_bf = cl.Pipeline([
    ('dev',   cl.Development(average=sel_avg, n_periods=sel_n)),
    ('tail',  cl.TailCurve(curve=sel_tail)),
    ('model', cl.BornhuetterFerguson(apriori=apriori_lr)),
])
pipe_cc = cl.Pipeline([
    ('dev',   cl.Development(average=sel_avg, n_periods=sel_n)),
    ('tail',  cl.TailCurve(curve=sel_tail)),
    ('model', cl.CapeCod()),
])

sw = build_exposure_triangle(exposure_series, tailed_dev)

voting = cl.VotingChainladder(
    estimators=[('cl', pipe_cl), ('bf', pipe_bf), ('cc', pipe_cc)],
    weights=W,
)
voting.fit(paid_triangle, sample_weight=sw)

blended_ult  = voting.ultimate_.to_frame(origin_as_datetime=False, keepdims=True)['paid'].values
blended_ibnr = voting.ibnr_.to_frame(origin_as_datetime=False, keepdims=True)['paid'].values
cl_ult = methods['Chain Ladder'][1]['ultimate'].values
bf_ult = methods['BF'][1]['ultimate'].values

blend_table = pd.DataFrame({
    'pct_paid':    pct_paid,
    'CL':          cl_ult,
    'BF':          bf_ult,
    'Blended':     blended_ult,
    'vs CL':       blended_ult - cl_ult,
    'vs BF':       blended_ult - bf_ult,
}, index=maturity.index[:n_origins])

blend_table.style.format({
    'pct_paid': '{:.1%}', 'CL': '{:,.0f}', 'BF': '{:,.0f}',
    'Blended': '{:,.0f}', 'vs CL': '{:+,.0f}', 'vs BF': '{:+,.0f}',
})


## Section 8 — Actual vs Expected: One-Period Emergence Test

**Purpose:** validate whether the selected development pattern accurately predicted the most recent quarter's claim emergence.

**Method:**
1. Fit each model on the truncated triangle (one quarter withheld).  
2. Extract expected cumulative at each AY's held-out development age from `full_expectation_`.  
3. Compute expected and actual single-quarter emergence.  
4. Report A/E ratio, MAE, and bias.

**Reading the results:**
- Aggregate A/E close to 1.0 → model was unbiased for this hold-out period  
- A/E consistently > 1.0 → model systematically underestimated development → pattern may be too low  
- Individual AY A/E outliers → data anomalies or method mismatch for that year  


In [ ]:
# ── 8a. Compute A/E ───────────────────────────────────────────────────────────
dev_trunc = cl.Development(average=sel_avg, n_periods=sel_n).fit_transform(paid_trunc)
sw_trunc  = build_exposure_triangle(exposure_series, dev_trunc)

ae_estimators = {
    'Chain Ladder': cl.Chainladder(),
    'BF':           cl.BornhuetterFerguson(apriori=apriori_lr),
    'Cape Cod':     cl.CapeCod(),
}

ae_results = {}
for method_name, estimator in ae_estimators.items():
    tail_t = cl.TailCurve(curve=sel_tail).fit_transform(dev_trunc)
    try:
        if method_name == 'Chain Ladder':
            model = estimator.fit(tail_t)
        else:
            model = estimator.fit(tail_t, sample_weight=sw_trunc)
    except Exception:
        continue

    fe_df = model.full_expectation_.to_frame(origin_as_datetime=False)
    rows  = []
    for origin, (age, actual_cum) in holdout_pairs.items():
        row_t = trunc_frame.loc[origin].dropna()
        if len(row_t) == 0:
            continue
        prior_cum  = float(row_t.iloc[-1])
        actual_emg = actual_cum - prior_cum
        if origin in fe_df.index and age in fe_df.columns and pd.notna(fe_df.loc[origin, age]):
            exp_emg = float(fe_df.loc[origin, age]) - prior_cum
        else:
            continue
        # A/E is only meaningful when expected emergence is positive
        ae = actual_emg / exp_emg if exp_emg > 0 else np.nan
        rows.append({'origin': origin, 'held_out_age': age,
                     'actual': actual_emg, 'expected': exp_emg,
                     'ae_ratio': ae, 'abs_error': abs(actual_emg - exp_emg)})
    ae_results[method_name] = pd.DataFrame(rows).set_index('origin')

# Print aggregate A/E
print('Aggregate A/E results — one-quarter holdout emergence:')
for method_name, df in ae_results.items():
    if df.empty:
        continue
    tot_a = df['actual'].sum()
    tot_e = df['expected'].sum()
    print(f'  {method_name:<20}  A/E = {tot_a/tot_e:.3f}  MAE = {df["abs_error"].mean():,.1f}')


In [ ]:
# ── 8b. AY-level detail ───────────────────────────────────────────────────────
if ae_results:
    first = next(iter(ae_results))
    ae_df = ae_results[first][['actual', 'expected', 'ae_ratio', 'abs_error']]
    print(f'AY-level A/E detail — {first}:')
    ae_df.style.format({
        'actual': '{:,.1f}', 'expected': '{:,.1f}',
        'ae_ratio': '{:.3f}', 'abs_error': '{:,.1f}',
    }).background_gradient(subset=['ae_ratio'], cmap='RdYlGn', vmin=0.7, vmax=1.3)


In [ ]:
# ── 8c. A/E scatter plot ──────────────────────────────────────────────────────
n_plots = len(ae_results)
if n_plots > 0:
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4), sharey=False)
    if n_plots == 1:
        axes = [axes]
    colors = ['steelblue', 'darkorange', 'seagreen']
    for ax, (mname, df), col in zip(axes, ae_results.items(), colors):
        if df.empty:
            continue
        ax.scatter(df['expected'], df['actual'], color=col, s=60, alpha=0.8, zorder=3)
        lo = min(df['expected'].min(), df['actual'].min()) * 0.9
        hi = max(df['expected'].max(), df['actual'].max()) * 1.1
        ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.4, label='A/E = 1.0')
        for _, r in df.iterrows():
            ax.annotate(str(r.name)[:4], (r['expected'], r['actual']),
                        textcoords='offset points', xytext=(4, 4), fontsize=7)
        ax.set_xlabel('Expected emergence')
        ax.set_ylabel('Actual emergence')
        ax.set_title(mname)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.suptitle('Actual vs Expected One-Quarter Emergence by AY', y=1.02)
    plt.tight_layout()
    plt.show()


## Section 9 — Reserve Range & Sensitivity

**Part A — Parameter sensitivity (GridSearch):** sweeps `average`, `n_periods`, and `tail` over plausible ranges. Scores by total IBNR. This is *model-assumption* uncertainty.

**Part B — Mack stochastic range:** `MackChainladder` provides prediction error under the Mack distribution-free assumptions. This is *statistical* uncertainty given fixed development factors.

The two types of uncertainty are complementary: Part A shows the range from judgment calls, Part B shows the range from statistical noise in the data.


In [ ]:
# ── 9a. Parameter sensitivity GridSearch ─────────────────────────────────────
pipe_s = cl.Pipeline([
    ('dev',   cl.Development()),
    ('tail',  cl.TailCurve()),
    ('model', cl.Chainladder()),
])

def score_ibnr(est):
    return float(np.nansum(est.named_steps['model'].ibnr_.values))

grid_s = cl.GridSearch(pipe_s, param_grid={
    'dev__average':   ['volume', 'simple'],
    'dev__n_periods': [3, 5, 7, -1],
    'tail__curve':    ['exponential', 'inverse_power'],
}, scoring=score_ibnr)
grid_s.fit(paid_triangle)

sens = grid_s.results_.rename(columns={'score': 'total_ibnr'}).sort_values('total_ibnr')
ibnr_lo  = sens['total_ibnr'].min()
ibnr_hi  = sens['total_ibnr'].max()
ibnr_sel = res_cl.ibnr_total

print(f'IBNR range across {len(sens)} combinations:')
print(f'  Low:        {ibnr_lo:>10,.0f}')
print(f'  Selected:   {ibnr_sel:>10,.0f}')
print(f'  High:       {ibnr_hi:>10,.0f}')
print(f'  Spread:     {ibnr_hi - ibnr_lo:>10,.0f}  ({(ibnr_hi - ibnr_lo) / ibnr_sel:.1%} of selected)')


In [ ]:
# ── 9b. Tornado chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: dot plot of all combinations
ax = axes[0]
colors = plt.cm.RdYlGn(np.linspace(0, 1, len(sens)))
ax.scatter(sens['total_ibnr'].reset_index(drop=True), range(len(sens)), c=colors, s=50, zorder=3)
ax.axvline(ibnr_sel, color='navy', linestyle='--', label='Selected')
ax.set_xlabel('Total IBNR')
ax.set_title('All Combinations Ranked by IBNR')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_yticks([])
ax.legend(fontsize=8)
ax.grid(True, axis='x', alpha=0.3)

# Right: within-group max spread per parameter
ax2 = axes[1]
param_names, spreads = [], []
for p in ['dev__average', 'dev__n_periods', 'tail__curve']:
    sp = sens.groupby(p)['total_ibnr'].apply(lambda s: s.max() - s.min()).max()
    param_names.append(p.split('__')[1])
    spreads.append(sp)
bars = ax2.barh(param_names, spreads, color='steelblue', alpha=0.85)
ax2.set_xlabel('Max within-group IBNR spread')
ax2.set_title('Which parameter drives the most uncertainty?')
ax2.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar, val in zip(bars, spreads):
    ax2.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
             f'{val:,.0f}', va='center', fontsize=9)
ax2.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── 9c. Mack stochastic prediction error ────────────────────────────────────
mack = cl.MackChainladder().fit(selected_dev)

mack_ibnr = float(np.nansum(mack.ibnr_.values))
# total_mack_std_err_ is a Triangle — extract the single scalar value
mack_se   = float(np.nansum(mack.total_mack_std_err_.values))
mack_cv   = mack_se / mack_ibnr if mack_ibnr else np.nan

# Log-normal approximation to percentiles
sigma2    = np.log(1 + mack_cv ** 2)
mu        = np.log(mack_ibnr) - 0.5 * sigma2
p75       = float(np.exp(mu + np.sqrt(sigma2) * 0.6745))
p90       = float(np.exp(mu + np.sqrt(sigma2) * 1.2816))

print('Mack stochastic prediction error:')
print(f'  Point estimate (IBNR):     {mack_ibnr:>12,.0f}')
print(f'  Total Mack std error:      {mack_se:>12,.0f}')
print(f'  Coefficient of variation:  {mack_cv:>12.1%}')
print(f'  ~75th pctile (log-normal): {p75:>12,.0f}')
print(f'  ~90th pctile (log-normal): {p90:>12,.0f}')


## Section 10 — Selected Reserve & Documentation

This section consolidates the analysis into a final deliverable: selected reserves with documented method rationale, key assumptions, and sensitivity flags.


In [ ]:
# ── 10a. Final method comparison ─────────────────────────────────────────────
rows = []
for mname, (res, _) in methods.items():
    rows.append({
        'Method':          mname,
        'Latest':          res.latest_reported_total,
        'Ultimate':        res.ultimate_total,
        'IBNR':            res.ibnr_total,
        'IBNR% of Latest': res.ibnr_total / res.latest_reported_total
                           if res.latest_reported_total else np.nan,
    })

rows.append({
    'Method':          'Blended (Voting)',
    'Latest':          res_cl.latest_reported_total,
    'Ultimate':        float(np.nansum(voting.ultimate_.values)),
    'IBNR':            float(np.nansum(voting.ibnr_.values)),
    'IBNR% of Latest': float(np.nansum(voting.ibnr_.values)) / res_cl.latest_reported_total,
})

final_df = pd.DataFrame(rows).set_index('Method')
final_df.style.format({
    'Latest': '{:,.0f}', 'Ultimate': '{:,.0f}',
    'IBNR': '{:,.0f}', 'IBNR% of Latest': '{:.1%}',
}).background_gradient(subset=['IBNR'], cmap='YlOrRd')


In [ ]:
# ── 10b. Assumption summary ───────────────────────────────────────────────────
assumptions = [
    ['Development — average method', sel_avg,
     'Cross-validation holdout winner'],
    ['Development — n_periods',      str(sel_n),
     'Cross-validation holdout winner'],
    ['Development — valuation drops', 'None',
     'Calendar-year screen showed no anomalous diagonals'],
    ['Tail estimator',               f'TailCurve(curve={sel_tail!r})',
     'Cross-validation selected; confirmed by tail sensitivity'],
    ['Tail factor',                  f'{selected_tail_f:.5f}',
     'Fitted from LDF progression beyond last observed age'],
    ['A priori loss ratio',          f'{apriori_lr:.2f}',
     'Back-solved proxy; replace with actual earned premium'],
    ['Berquist-Sherman adjustment',  'None',
     'Paid/reported LDF ratio screen showed no material trend'],
    ['Cross-validation holdout MAE', f"{best_row['holdout_mae']:.2f}",
     'Best configuration; see Section 4 for full ranking'],
    ['Mack CV (stat uncertainty)',   f'{mack_cv:.1%}',
     'Process + parameter risk; distribution-free estimate'],
    ['Method range (Low / High)',
     f'{ibnr_lo:,.0f} / {ibnr_hi:,.0f}',
     'Across all parameter combinations; see Section 9'],
]

pd.DataFrame(assumptions, columns=['Assumption', 'Value', 'Rationale'])\
  .set_index('Assumption')\
  .style.set_properties(**{'text-align': 'left'})


## Closing Notes — Adapting This Template to Your Data

1. **Section 0** — Replace `paid_triangle` and `reported_triangle` with your own Triangle objects.  
   Both must be cumulative and cover the same origins and development ages.

2. **Section 1** — Check the maturity profile.  
   AYs with % paid < 20%: BF/Benktander will dominate.  
   AYs with % paid > 90%: chain ladder is sufficient; BF adds minimal value.

3. **Section 4 (cross-validation)** — Holdout is 3 months (one quarter).  
   For annual triangles change `pd.DateOffset(months=3)` to `pd.DateOffset(months=12)`.

4. **Section 6 (exposure)** — Replace `ep_proxy` with actual earned premium by accident year.  
   This affects BF, Benktander, Cape Cod, and Expected Loss ultimates directly.

5. **Section 7 (blending weights)** — The `pct_paid > 0.70` thresholds are illustrative.  
   Adjust to your line of business and view of when the chain ladder becomes credible.

6. **Section 3 (Berquist-Sherman)** — Uncomment the template and add count triangles  
   (`closed_count_triangle`, `reported_count_triangle`) when Screen 3 flags a shift.

---

**Further extensions:**
- `cl.BootstrapODPSample` — over-dispersed Poisson bootstrap for a full reserve distribution  
- `cl.MunichAdjustment` — bivariate paid/reported adjustment (alternative to Berquist-Sherman)  
- `cl.ClarkLDF` — parametric MLE curve fitting as an alternative to period-by-period averaging  
- Multi-segment triangles — all utilities handle multi-index triangles automatically  
